In [1]:
# ############################################################
# 성능개선 예시 — baseline -> 교차검증 -> 정규화 -> 튜닝 -> 파이프라인
# ############################################################
# ------------------------------------------------------------
# [목적] 도구 + 데이터 준비 (당뇨 진행도 예측, 회귀)
# ------------------------------------------------------------
from sklearn.datasets import load_diabetes        # 당뇨 데이터 (숫자 예측)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score
X, y = load_diabetes(return_X_y=True)             # 특성 X, 정답 y
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0)  # 8:2

In [2]:
# ------------------------------------------------------------
# [목적] ① baseline — 딱 한 번 나눠 점수 재기 (출발점)
# ------------------------------------------------------------
lin = LinearRegression().fit(X_tr, y_tr)          # 학습
print('단일 분할 R2:', round(r2_score(y_val, lin.predict(X_val)), 4))  # 한 번 나눈 점수(운에 좌우)

단일 분할 R2: 0.3322


In [3]:
# ------------------------------------------------------------
# [목적] ② 교차검증 — 5번 나눠 평균 (믿을 수 있는 '진짜 실력')
# ------------------------------------------------------------
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')  # 5조각으로 5번 채점
print('5개 점수:', scores.round(3))               # 조각마다 점수가 출렁임
print('평균 R2 :', round(scores.mean(), 4))       # 평균이 대표 성적

5개 점수: [0.43  0.523 0.483 0.426 0.55 ]
평균 R2 : 0.4823


In [4]:
# ------------------------------------------------------------
# [목적] ③ 정규화 모델 비교 — Ridge / Lasso (과적합 억제)
# ------------------------------------------------------------
for name, model in [('Ridge', Ridge(alpha=1)), ('Lasso', Lasso(alpha=0.1))]:  # 두 모델
    s = cross_val_score(model, X, y, cv=5, scoring='r2').mean()   # 교차검증 평균
    print(name, round(s, 4))                       # 정규화 세기에 따라 점수 달라짐

Ridge 0.4102
Lasso 0.4795


In [5]:
# ------------------------------------------------------------
# [목적] ④ 튜닝 — GridSearchCV로 alpha 자동 탐색 (최고 설정 찾기)
# ------------------------------------------------------------
grid = GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10, 100]},   # alpha 후보들
                    cv=5, scoring='r2')            # 5겹 교차검증으로 채점
grid.fit(X_tr, y_tr)                               # 후보를 모두 시도
print('최적 alpha :', grid.best_params_)           # 가장 좋았던 설정
print('최적 CV R2 :', round(grid.best_score_, 4))  # 그때의 점수

최적 alpha : {'alpha': 0.01}
최적 CV R2 : 0.5267


In [6]:
# ------------------------------------------------------------
# [목적] ⑤ 파이프라인 — 스케일링+모델을 한 상자로 (누수 방지)
# ------------------------------------------------------------
pipe = make_pipeline(StandardScaler(), Ridge(alpha=1))   # 전처리+모델 묶기
print('파이프라인 CV R2:',
      round(cross_val_score(pipe, X, y, cv=5, scoring='r2').mean(), 4))  # 교차검증 중 스케일링도 안전

파이프라인 CV R2: 0.4822


In [7]:
# ============================================================
# [결과 해석]
#  · 단일 분할 R2 ~ 0.33  ->  교차검증 평균 ~ 0.48
#      = 한 번 나눈 점수(0.33)는 '운이 나쁜' 조각이었고, 5번 평균이 더 믿을 만함
#  · Ridge 0.41 / Lasso 0.48 -> 정규화 세기에 따라 성능이 달라짐
#  · GridSearch 최적 alpha=0.01, CV R2 ~ 0.53 -> 자동 탐색으로 최고 성능
#  · 파이프라인은 교차검증 중에도 스케일링 정보가 새지 않게 막아줌
#  · 교훈: '한 번 점수'에 속지 말고 교차검증 + 튜닝으로 판단!
# ============================================================